In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%ls "/content/drive/My Drive/PropInsight/corpus/"

glossary_property.txt  glossaryV3.csv  SGPropertyDomain/


In [ ]:
# === SG Property Glossary → Full NLP Corpus
# === Singapore Real Estate Glossary → Domain Corpus & Dictionary ===
# Input: raw text glossary (your collected file)
# Output: glossary.jsonl, glossary.csv, spacy_entityruler_patterns.jsonl, regex_patterns.jsonl, vocab/*.txt
# Safe to run multiple times — idempotent, merges duplicates by canonical term.

# Input : /content/drive/My Drive/PropInsight/corpus/glossary_property.txt
# Output: /content/drive/My Drive/PropInsight/corpus/SGPropertyDomain/*

import re, json, csv, unicodedata
from pathlib import Path
from collections import defaultdict, Counter

# ----- Paths -----
SRC = Path("/content/drive/My Drive/PropInsight/corpus/glossary_property.txt")
OUT = Path("/content/drive/My Drive/PropInsight/corpus/SGPropertyDomain")
VOC = OUT / "vocab"
OUT.mkdir(parents=True, exist_ok=True)
VOC.mkdir(parents=True, exist_ok=True)

# ========== UTILITIES ==========
def clean(s: str) -> str:
    """Unicode normalize, strip zero-width/NBSP, trim."""
    if not s: return ""
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u200b\u200c\u200d\uFEFF\u00A0]", "", s)
    return s.strip()

def norm_key(s: str) -> str:
    """
    Canonical key for deduplication: lowercase, collapse spaces/hyphens, strip punctuation.
    Makes 'ABSD', 'absd ', 'A B S D' all the same.
    """
    s = clean(s).lower()
    s = re.sub(r"[\-–—_/]+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)   # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()
    return s

ENTRY_LINE = re.compile(r"^\s*([A-Z0-9#][^:]{0,160}?)\s*:\s*(.+)$")

def parse_entries(text: str):
    """Parse TERM: definition … (multi-line defs until next TERM:)."""
    entries, cur_term, cur_def = [], None, []
    for raw in text.splitlines():
        line = clean(raw)
        if not line:
            continue

        # skip obvious headings like "#RES/Web" or "Glossary ..."
        if line.startswith("#"):
            continue
        if line.lower().startswith("glossary"):
            continue

        m = ENTRY_LINE.match(line)
        if m:
            if cur_term:
                entries.append((cur_term, " ".join(cur_def).strip()))
            cur_term = m.group(1).strip()
            cur_def  = [m.group(2).strip()]
        else:
            if cur_term:
                cur_def.append(line)
    if cur_term:
        entries.append((cur_term, " ".join(cur_def).strip()))
    return entries



def make_id(term: str) -> str:
    return "term." + re.sub(r"[^a-z0-9]+", "_", term.lower()).strip("_")

def make_regex(term: str) -> str:
    t = re.escape(term)
    t = t.replace("\\ ", "\\s+").replace("\\-", "[-–—]?")
    return rf"(?i)\b{t}\b"

def guess_category(term: str, definition: str) -> str:
    s = (term + " " + definition).lower()
    if re.search(r"\bhdb|bto|mop|jumbo|executive flat|maisonette|ec\b", s): return "HDB"
    if re.search(r"\b(bungalow|terrace|semi|semi\-?d|shophouse|condo|minium|walk\-up|cluster|town ?house|private apartment|black and white)\b", s): return "PropertyType"
    if re.search(r"\b(absd|bsd|ssd|stamp duty|development charge|lbc|dc)\b", s): return "Tax&Duty"
    if re.search(r"\b(top|csc|tenancy|s&p|loi|lpa|poa|agreement|otp)\b", s): return "Legal&Docs"
    if re.search(r"\b(ltv|tdsr|msr|sibor|sora|sor|apr|emi|mortgage|refinanc|repric)\b", s): return "Finance&Rates"
    if re.search(r"\b(en\-?bloc|collective sale|valuation|appraisal|foreclosure|decoupling)\b", s): return "Process"
    if re.search(r"\b(psf|per square foot|cma|annual value|cap(italisation)? rate)\b", s): return "MarketMetric"
    if re.search(r"\b(cea|hdb|ura|sla|ldau|re?it)\b", s): return "Agency&Authority"
    if re.search(r"\b(freehold|leasehold|tenure|99[- ]?year|999[- ]?year)\b", s): return "Tenure"
    if re.search(r"\b(ccr|rcr|ocr|core central|rest of central|outside central)\b", s): return "Region"
    if re.search(r"\b(fogging|pest control|service charge|home insurance|fire insurance)\b", s): return "Maintenance"
    return "Other"

def dedup_list(xs):
    """Order-preserving unique."""
    seen, out = set(), []
    for x in xs:
        k = (x or "").strip()
        if k and k not in seen:
            seen.add(k); out.append(k)
    return out


import math

def _safe_str(x):
    """Return clean string; handle NaN/None safely."""
    if x is None:
        return ""
    if isinstance(x, float):
        if math.isnan(x):
            return ""
    return str(x)



# ========== LOAD & PARSE ==========
text = clean(SRC.read_text(encoding="utf-8"))
pairs = parse_entries(text)


# ========= ADD: ingest structured CSV with [HEADING-H2]/[PARAGRAPH]/[LIST-ITEM] =========
import pandas as _pd

CSV_PATH = Path("/content/drive/My Drive/PropInsight/corpus/glossaryV3.csv")

TAG_HEADING = re.compile(r"^\s*\[(?:HEADING|HEADING\-H2)\]\s*(.+?)\s*$", flags=re.IGNORECASE)
TAG_PARA    = re.compile(r"^\s*\[(?:PARAGRAPH|PARAGRAGH)\]\s*(.+?)\s*$", flags=re.IGNORECASE)
TAG_ITEM    = re.compile(r"^\s*\[(?:LIST\-ITEM|ITEM)\]\s*(.+?)\s*$", flags=re.IGNORECASE)

def parse_content_sections(content: str):
    """
    Turn a single 'content' blob into (term, definition) pairs:
      - New section on [HEADING-H2]
      - Accumulate following [PARAGRAPH]/[PARAGRAGH]/[LIST-ITEM] lines into a definition
      - Stop when next heading appears
    Returns: List[(term, definition)]
    """
    if not content:
        return []
    sections = []
    cur_term, cur_bits = None, []

    # split lines; keep only non-empty after clean

    for raw in content.splitlines():
        line = clean(raw)
        if not line:
            continue

        m_h = TAG_HEADING.match(line)
        if m_h:
            # flush previous
            if cur_term:
                # join with bullet for list items already in cur_bits
                defin = " ".join(cur_bits).strip()
                sections.append((cur_term, defin))
            cur_term = m_h.group(1).strip()
            cur_bits = []
            continue

        m_p = TAG_PARA.match(line)
        if m_p and cur_term:
            cur_bits.append(m_p.group(1).strip())
            continue

        m_i = TAG_ITEM.match(line)
        if m_i and cur_term:
            # keep bullet flavor in plain text (helps readability)
            cur_bits.append("• " + m_i.group(1).strip())
            continue

        # If a stray line without a tag appears under a current term, treat as paragraph
        if cur_term and not (TAG_HEADING.match(line) or TAG_PARA.match(line) or TAG_ITEM.match(line)):
            cur_bits.append(line)

    if cur_term:
        defin = " ".join(cur_bits).strip()
        sections.append((cur_term, defin))

    return sections

def pairs_from_csv(csv_path: Path):
    pairs_extra = []
    if not csv_path.exists():
        return pairs_extra
    df = _pd.read_csv(csv_path)
    # be defensive: only rows with content
    for _, row in df.iterrows():
        content = _safe_str(row.get("content", ""))
        title   = clean(_safe_str(row.get("title_page", "")))
        url     = clean(_safe_str(row.get("url", "")))
        term_index = clean(_safe_str(row.get("term_index", "")))

        secs = parse_content_sections(content)
        for (term, defin) in secs:
            # Optionally enrich definition with provenance
            tail = []
            if title:
                tail.append(f"Source: {title}")
            if url:
                tail.append(url)
            if term_index:
                tail.append(f"(index: {term_index})")
            if tail:
                defin = (defin + "  " if defin else "") + "  ".join(tail)

            pairs_extra.append((term, defin))
    return pairs_extra

# load & merge
pairs_from_structured = pairs_from_csv(CSV_PATH)
print(f"📥 From CSV: extracted {len(pairs_from_structured)} (term, def) pairs")
pairs += pairs_from_structured




# ========== DEDUP: MERGE BY CANONICAL KEY ==========
# 1) bucket by canonical key (handles case/spacing/punct)
bucket = defaultdict(list)
for term, defin in pairs:
    t, d = clean(term), clean(defin)
    if not t: continue
    # If term looks like “ABSD (Additional Buyer’s Stamp Duty)”, keep both forms
    # and use the more descriptive side for the key to encourage merging.
    m = re.match(r"^\s*(.+?)\s*\(([^)]+)\)\s*$", t)
    if m:
        left, right = clean(m.group(1)), clean(m.group(2))
        key = norm_key(right if len(right) > len(left) else left)
    else:
        key = norm_key(t)
    bucket[key].append((t, d))

dup_groups = {k:v for k,v in bucket.items() if len(v) > 1}

# 2) build unified records per bucket
records = []
id_set = set()
canonical_terms = set()
#for key, group in bucket.items():
#    # display term: choose longest (more descriptive) surface
#    display = max((g[0] for g in group), key=len)
#    definition_list = dedup_list([g[1] for g in group if g[1]])
#    definition = definition_list[0] if definition_list else ""
#    # aliases: start with surfaces from the bucket (except display)
#    alias_raw = [g[0] for g in group if g[0] != display]
#
#    # add aliases from parentheses: "ABSD (Additional ...)" → ["ABSD", "Additional ..."]
#    m_disp = re.match(r"^\s*(.+?)\s*\(([^)]+)\)\s*$", display)
#    if m_disp:
#        alias_raw += [m_disp.group(1).strip(), m_disp.group(2).strip()]
#
#    # contiguous all-caps token in display, 2–6 chars (e.g., "... (BSD)")
#    m_ac = re.search(r"\b([A-Z]{2,6})\b", display)
#    if m_ac:
#        alias_raw.append(m_ac.group(1))
#
#    # derive initials from Title Case phrase (e.g., "Core Central Region" → "CCR")
#    toks = re.findall(r"[A-Za-z]+", display)
#    if 2 <= len(toks) <= 5 and all(tok[:1].isupper() for tok in toks):
#        initials = "".join(tok[0] for tok in toks)
#        if 2 <= len(initials) <= 6:
#            alias_raw.append(initials)
#
#    # split slash variants (e.g., "TOP/CSC" → "TOP","CSC")
#    for tok in [display] + alias_raw[:]:
#        if "/" in tok:
#            alias_raw += [p.strip() for p in tok.split("/") if p.strip()]
#
#    # collapse near-dupes by normalized key and remove the display itself
#    alias_by_norm = {}
#    for a in alias_raw:
#        nk = norm_key(a)
#        if nk and nk != norm_key(display) and nk not in alias_by_norm:
#            alias_by_norm[nk] = a
#    aliases = dedup_list(alias_by_norm.values())
#
#    cat = guess_category(display, definition)
#
#    # ensure term canonical uniqueness (by norm_key)
#    term_norm = norm_key(display)
#    if term_norm in canonical_terms:
#        # pick an alternative representative if needed
#        # (shouldn’t happen due to bucketing, but guard anyway)
#        display = next((a for a in aliases if norm_key(a) not in canonical_terms), display)
#        term_norm = norm_key(display)
#    canonical_terms.add(term_norm)
#
#    rid_base = make_id(display)
#    rid = rid_base
#    if rid in id_set:
#        # enforce unique IDs deterministically
#        i = 2
#        while f"{rid_base}_{i}" in id_set:
#            i += 1
#        rid = f"{rid_base}_{i}"
#    id_set.add(rid)
#
#    # patterns: one phrase + one regex (regex dedup implicit)
#    patterns = [{"type":"phrase","value":display}]
#    if len(display) > 3:
#        patterns.append({"type":"regex","value": make_regex(display)})
#
#    records.append({
#        "id": rid,
#        "term": display,
#        "norm": display.lower(),
#        "category": cat,
#        "definition": definition,
#        "aliases": aliases,
#        "patterns": patterns,
#        "source": "PropInsight Glossary",
#        "version": 1
#    })


for key, group in bucket.items():
    # display term: choose longest (more descriptive) surface
    display = max((g[0] for g in group), key=len)
    definition_list = dedup_list([g[1] for g in group if g[1]])
    definition = definition_list[0] if definition_list else ""

    # --- BEGIN: normalize question headings + flip acronym form ---
    def normalize_question_term(t: str) -> str:
      s = (t or "").strip()
      s = re.sub(r"^\s*what\s+is\s+the\s+", "", s, flags=re.IGNORECASE)
      s = re.sub(r"^\s*what\s+is\s+", "", s, flags=re.IGNORECASE)
      s = re.sub(r"^\s*what\s+are\s+the\s+", "", s, flags=re.IGNORECASE)
      s = re.sub(r"^\s*what\s+are\s+", "", s, flags=re.IGNORECASE)
      s = s.rstrip("?").strip()
      return s

    display = normalize_question_term(display)

    # Prefer “Long (ACRONYM)” instead of “ACRONYM (Long)”
    flip = re.match(r"^\s*([A-Z]{2,6})\s*\(([^)]+)\)\s*$", display)
    if flip and not flip.group(2).isupper():
        display = f"{flip.group(2).strip()} ({flip.group(1).strip()})"
    # --- END ---

    # === CLEAN alias generation ===
    alias_raw = [g[0] for g in group if g[0] != display]

    # Extract acronym from parentheses if present (e.g., "Core Central Region (CCR)")
    m_disp = re.match(r"^\s*(.+?)\s*\(([^)]+)\)\s*$", display)
    acronym = None
    if m_disp:
        left, right = m_disp.group(1).strip(), m_disp.group(2).strip()
        alias_raw += [left, right]
        if right.isupper() and 2 <= len(right) <= 6:
            acronym = right

    # Contiguous ALL-CAPS token in display (2–6 chars), if not same as known acronym
    m_ac = re.search(r"\b([A-Z]{2,6})\b", display)
    if m_ac and m_ac.group(1) != acronym:
        alias_raw.append(m_ac.group(1))

    # Derive initials only from Title-Case words (skip ALL-CAPS to avoid CCCR/OOCR)
    toks = re.findall(r"[A-Za-z]+", display)
    title_words = [tok for tok in toks if tok[:1].isupper() and not tok.isupper()]
    if 2 <= len(title_words) <= 6:
        initials = "".join(tok[0] for tok in title_words)
        if 2 <= len(initials) <= 6 and initials != acronym:
            if initials not in alias_raw:
                alias_raw.append(initials)

    # Split slash variants (e.g., "TOP/CSC" → "TOP","CSC")
    for tok in [display] + alias_raw[:]:
        if "/" in tok:
            alias_raw += [p.strip() for p in tok.split("/") if p.strip()]

    # Collapse near-dupes by normalized key and remove the display itself
    alias_by_norm = {}
    for a in alias_raw:
        nk = norm_key(a)
        if nk and nk != norm_key(display) and nk not in alias_by_norm:
            alias_by_norm[nk] = a
    aliases = dedup_list(alias_by_norm.values())
    # === END CLEAN alias generation ===

    cat = guess_category(display, definition)

    # ensure term canonical uniqueness (by norm_key)
    term_norm = norm_key(display)
    if term_norm in canonical_terms:
        # pick an alternative representative if needed
        display = next((a for a in aliases if norm_key(a) not in canonical_terms), display)
        term_norm = norm_key(display)
    canonical_terms.add(term_norm)

    rid_base = make_id(display)
    rid = rid_base
    if rid in id_set:
        # enforce unique IDs deterministically
        i = 2
        while f"{rid_base}_{i}" in id_set:
            i += 1
        rid = f"{rid_base}_{i}"
    id_set.add(rid)

    # patterns: one phrase + one regex (regex dedup implicit)
    patterns = [{"type": "phrase", "value": display}]
    if len(display) > 3:
        patterns.append({"type": "regex", "value": make_regex(display)})

    records.append({
        "id": rid,
        "term": display,
        "norm": display.lower(),
        "category": cat,
        "definition": definition,
        "aliases": aliases,
        "patterns": patterns,
        "source": "PropInsight Glossary",
        "version": 1
    })


# ========== WRITE ARTIFACTS ==========
# 1) JSONL
(OUT / "glossary.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in records),
    encoding="utf-8"
)

# 2) CSV
with (OUT / "glossary.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id","term","category","definition","aliases"])
    for r in records:
        w.writerow([r["id"], r["term"], r["category"], r["definition"], "|".join(r["aliases"])])

# 3) spaCy patterns (phrases only for speed)
#spacy_patterns = []
#for r in records:
#    lab = r["category"].upper()
#    spacy_patterns.append({"label": lab, "pattern": r["term"]})
#    for a in r["aliases"][:8]:
#        spacy_patterns.append({"label": lab, "pattern": a})
#(OUT / "spacy_entityruler_patterns.jsonl").write_text(
#    "\n".join(json.dumps(p, ensure_ascii=False) for p in spacy_patterns),
#    encoding="utf-8"
#)

# 3) spaCy patterns (phrases; include ALL aliases)
spacy_patterns = []
for r in records:
    lab = r["category"].upper()
    spacy_patterns.append({"label": lab, "pattern": r["term"]})
    for a in r["aliases"]:   # <-- no slicing
        spacy_patterns.append({"label": lab, "pattern": a})

(OUT / "spacy_entityruler_patterns.jsonl").write_text(
    "\n".join(json.dumps(p, ensure_ascii=False) for p in spacy_patterns),
    encoding="utf-8"
)


# 4) Regex patterns
#regex_patterns = []
#for r in records:
#    for p in r["patterns"]:
#        if p["type"] == "regex":
#            regex_patterns.append({"id": r["id"], "regex": p["value"]})
#(OUT / "regex_patterns.jsonl").write_text(
#    "\n".join(json.dumps(p, ensure_ascii=False) for p in regex_patterns),
#    encoding="utf-8"
#)

# 4) Regex patterns (DISPLAY + ALIASES)
regex_patterns = []
def _make_regex(t):
    import re as _re
    tt = _re.escape(t).replace("\\ ", "\\s+").replace("\\-", "[-–—]?")
    return rf"(?i)\b{tt}\b"

for r in records:
    if len(r["term"]) > 1:
        regex_patterns.append({"id": r["id"], "regex": _make_regex(r["term"])})
    for a in r["aliases"]:
        if len(a) > 1:
            regex_patterns.append({"id": r["id"], "regex": _make_regex(a)})

(OUT / "regex_patterns.jsonl").write_text(
    "\n".join(json.dumps(p, ensure_ascii=False) for p in regex_patterns),
    encoding="utf-8"
)


# 5) Aspect vocab (per category)
cats = defaultdict(list)
for r in records:
    cats[r["category"]].append(r["term"])
for c, terms in cats.items():
    (VOC / f"{c}.txt").write_text("\n".join(sorted(set(terms))), encoding="utf-8")

# 6) Optional plain-text corpus
with (OUT / "sg_property_corpus.txt").open("w", encoding="utf-8") as f:
    for r in records:
        if r["definition"]:
            f.write(f"{r['term']} - {re.sub(r'\\s+', ' ', r['definition']).strip()}\n")
        else:
            f.write(f"{r['term']} - \n")

# ========== VALIDATION & REPORT ==========
# Unique IDs & canonical terms
id_dupes = len(id_set) != len(records)
term_norms = [norm_key(r["term"]) for r in records]
term_dupe_counts = [t for t,c in Counter(term_norms).items() if c > 1]

print(f"✅ Built {len(records)} unique terms → {OUT}")
print(f"🔎 Merged duplicate groups: {len(dup_groups)}")
if id_dupes or term_dupe_counts:
    print("⚠️ Duplication check: IDs unique?", not id_dupes, "| Canonical terms unique?", len(term_dupe_counts)==0)
else:
    print("✅ No duplicate IDs or canonical terms.")
print("Files written:")
for p in sorted(OUT.glob("*")):
    if p.is_file():
        print("  -", p.name)


📥 From CSV: extracted 971 (term, def) pairs
✅ Built 1025 unique terms → /content/drive/My Drive/PropInsight/corpus/SGPropertyDomain
🔎 Merged duplicate groups: 29
✅ No duplicate IDs or canonical terms.
Files written:
  - glossary.csv
  - glossary.jsonl
  - regex_patterns.jsonl
  - sg_property_corpus.txt
  - spacy_entityruler_patterns.jsonl


In [ ]:
# ==== SG Property Domain: Mini Knowledge-Base Search ====
# Uses your glossary.jsonl to provide:
#  - exact lookup (by term)
#  - keyword/regex search
#  - fuzzy search (RapidFuzz)
#  - optional semantic search (Sentence-Transformers)

# --- config ---
BASE = "/content/drive/My Drive/PropInsight/corpus/SGPropertyDomain"
GLOSSARY_JSONL = f"{BASE}/glossary.jsonl"

import json, re, unicodedata, math
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

# Optional deps (safe to run repeatedly)
!pip -q install rapidfuzz > /dev/null
try:
    import sentence_transformers  # noqa
    _HAS_ST = True
except Exception:
    _HAS_ST = False

from rapidfuzz import fuzz, process

# ---------------- utilities ----------------
def _clean(s: str) -> str:
    if s is None: return ""
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u200b\u200c\u200d\uFEFF\u00A0]", "", s)  # zero-width & nbsp
    return s.strip()

def _load_jsonl(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")
    out = []
    for line in p.read_text(encoding="utf-8").splitlines():
        if line.strip():
            out.append(json.loads(line))
    return out

@dataclass
class KBEntry:
    id: str
    term: str
    category: str
    definition: str
    aliases: List[str]

# --------------- load glossary ---------------
raw = _load_jsonl(GLOSSARY_JSONL)
KB: List[KBEntry] = []
for r in raw:
    KB.append(
        KBEntry(
            id=r.get("id",""),
            term=_clean(r.get("term","")),
            category=_clean(r.get("category","Other")),
            definition=_clean(r.get("definition","")),
            aliases=[_clean(a) for a in r.get("aliases",[])]
        )
    )

# quick indexes
TERMS_LOWER = {e.term.lower(): e for e in KB}
ALIASES_TO_ID = {}
for e in KB:
    for a in e.aliases:
        ALIASES_TO_ID[a.lower()] = e.id
ID_TO_ENTRY = {e.id: e for e in KB}

print(f"✅ Loaded {len(KB)} glossary entries from {GLOSSARY_JSONL}")

# --------------- exact lookup ---------------
def kb_lookup(term: str) -> KBEntry | None:
    """Exact term or alias (case-insensitive)."""
    t = term.strip().lower()
    if t in TERMS_LOWER:
        return TERMS_LOWER[t]
    if t in ALIASES_TO_ID:
        return ID_TO_ENTRY.get(ALIASES_TO_ID[t])
    return None

# --------------- keyword / regex search ---------------
def kb_keyword_search(query: str, top_k: int = 10, regex: bool = False) -> List[Tuple[KBEntry, float]]:
    """
    If regex=False: simple case-insensitive 'query in text' over term/aliases/definition.
    If regex=True: treat query as a regex (re.IGNORECASE).
    Returns list of (entry, score) where score is a crude hit count / relevance.
    """
    q = query.strip()
    if not q:
        return []
    results = []
    if not regex:
        ql = q.lower()
        for e in KB:
            hay = " ".join([e.term, " ".join(e.aliases), e.definition]).lower()
            if ql in hay:
                # crude score: occurrences count
                score = hay.count(ql)
                results.append((e, float(score)))
    else:
        rx = re.compile(q, flags=re.IGNORECASE)
        for e in KB:
            hay = " ".join([e.term] + e.aliases + [e.definition])
            hits = rx.findall(hay)
            if hits:
                results.append((e, float(len(hits))))
    # sort by score desc, then shorter term first
    results.sort(key=lambda x: (-x[1], len(x[0].term)))
    return results[:top_k]

# --------------- fuzzy search (RapidFuzz) ---------------
# Build a corpus of candidate strings (term + aliases)
CANDIDATES = []
for e in KB:
    CANDIDATES.append((e.id, e.term))
    for a in e.aliases:
        CANDIDATES.append((e.id, a))

def kb_fuzzy_search(query: str, top_k: int = 10, scorer=fuzz.WRatio) -> List[Tuple[KBEntry, float, str]]:
    """
    Fuzzy match over term + aliases. Returns (entry, score, matched_surface).
    """
    pool = [cand[1] for cand in CANDIDATES]
    matches = process.extract(query, pool, scorer=scorer, limit=top_k)
    out = []
    for surface, score, idx in matches:
        entry_id = CANDIDATES[idx][0]
        out.append((ID_TO_ENTRY[entry_id], float(score), surface))
    # Deduplicate by entry id keeping best score
    best = {}
    for e, sc, surf in out:
        if e.id not in best or sc > best[e.id][0]:
            best[e.id] = (sc, surf)
    ranked = [(ID_TO_ENTRY[i], sc, surf) for i,(sc, surf) in best.items()]
    ranked.sort(key=lambda x: -x[1])
    return ranked[:top_k]

# --------------- semantic search (optional) ---------------
_ST_MODEL = None
_DOC_EMB = None

def kb_enable_semantic(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    """Load embedding model and precompute embeddings for term + definition."""
    global _ST_MODEL, _DOC_EMB
    from sentence_transformers import SentenceTransformer
    _ST_MODEL = SentenceTransformer(model_name)
    texts = [f"{e.term}. {e.definition}" if e.definition else e.term for e in KB]
    _DOC_EMB = _ST_MODEL.encode(texts, normalize_embeddings=True)
    print(f"🔎 Semantic search enabled with {model_name}. Cached {len(texts)} embeddings.")

def kb_semantic_search(query: str, top_k: int = 10) -> List[Tuple[KBEntry, float]]:
    """Cosine similarity over SBERT embeddings."""
    if _ST_MODEL is None or _DOC_EMB is None:
        raise RuntimeError("Semantic search not enabled. Call kb_enable_semantic() first.")
    qv = _ST_MODEL.encode([query], normalize_embeddings=True)[0]
    # cosine sim = dot since normalized
    import numpy as np
    sims = np.dot(_DOC_EMB, qv)
    idx = sims.argsort()[-top_k:][::-1]
    out = []
    for i in idx:
        out.append((KB[i], float(sims[i])))
    return out

# --------------- pretty print helpers ---------------
def show(entries):
    for i,(e,score,*rest) in enumerate(entries, start=1):
        surf = f" | match='{rest[0]}'" if rest else ""
        print(f"{i:2d}. [{e.category}] {e.term}{surf}")
        if e.aliases:
            print("    aka:", ", ".join(e.aliases[:6]))
        if e.definition:
            print("    def:", e.definition[:240] + ("..." if len(e.definition)>240 else ""))
        print(f"    score: {score:.3f}\n")

def show_one(entry: KBEntry | None):
    if not entry:
        print("No match.")
        return
    print(f"[{entry.category}] {entry.term}  (id: {entry.id})")
    if entry.aliases:
        print("aka:", ", ".join(entry.aliases))
    print(entry.definition)

print("💡 Ready. Try:")
print("   show(kb_keyword_search('absd', top_k=5))")
print("   show(kb_fuzzy_search('buyers stamp', top_k=5))")
print("   e = kb_lookup('BTO'); show_one(e)")
print("   # Optional semantic:")
print("   # kb_enable_semantic(); show(kb_semantic_search('taxes when buying second property', 5))")


✅ Loaded 1025 glossary entries from /content/drive/My Drive/PropInsight/corpus/SGPropertyDomain/glossary.jsonl
💡 Ready. Try:
   show(kb_keyword_search('absd', top_k=5))
   show(kb_fuzzy_search('buyers stamp', top_k=5))
   e = kb_lookup('BTO'); show_one(e)
   # Optional semantic:
   # kb_enable_semantic(); show(kb_semantic_search('taxes when buying second property', 5))


#Sanity summary

In [ ]:
from pathlib import Path
import json, pandas as pd, collections

BASE = Path("/content/drive/My Drive/PropInsight/corpus/SGPropertyDomain")
recs = [json.loads(l) for l in (BASE/"glossary.jsonl").read_text(encoding="utf-8").splitlines()]
print("Terms:", len(recs))
print("Files:", [p.name for p in sorted(BASE.glob("*")) if p.is_file()])

# category counts + empties
cnt = collections.Counter(r["category"] for r in recs)
print("Category counts:", dict(cnt))
print("Empty definitions:", sum(1 for r in recs if not r["definition"].strip()))


Terms: 1025
Files: ['glossary.csv', 'glossary.jsonl', 'regex_patterns.jsonl', 'sg_property_corpus.txt', 'spacy_entityruler_patterns.jsonl']
Category counts: {'Other': 509, 'Tax&Duty': 18, 'PropertyType': 48, 'HDB': 180, 'MarketMetric': 17, 'Agency&Authority': 38, 'Legal&Docs': 93, 'Process': 25, 'Maintenance': 3, 'Tenure': 9, 'Finance&Rates': 81, 'Region': 4}
Empty definitions: 0


#Try spaCy tagging (zero training)

In [ ]:
import spacy
nlp = spacy.blank("en")
ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True, "phrase_matcher_attr": "LOWER"})
ruler.from_disk("/content/drive/My Drive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl")

doc = nlp("ABSD and BSD apply in CCR; many BTO flats hit MOP. TOP / CSC appear.")
print([(e.text, e.label_) for e in doc.ents])



[('ABSD', 'TAX&DUTY'), ('BSD', 'TAX&DUTY'), ('CCR', 'REGION'), ('BTO', 'HDB'), ('MOP', 'HDB'), ('TOP', 'LEGAL&DOCS'), ('CSC', 'LEGAL&DOCS')]


#Regex rule hit-test

In [ ]:
import re, json, pathlib
BASE = pathlib.Path("/content/drive/My Drive/PropInsight/corpus/SGPropertyDomain")
rx = [json.loads(l)["regex"] for l in (BASE/"regex_patterns.jsonl").read_text(encoding="utf-8").splitlines()]
text = "Buyer’s Stamp Duty (BSD) and ABSD impact purchases; PSF differs in OCR vs CCR."
print("Regex hits:", sum(bool(re.search(r, text)) for r in rx))


Regex hits: 23


In [ ]:
from pathlib import Path
import json

pat_path = Path("/content/drive/My Drive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl")
lines = pat_path.read_text(encoding="utf-8").splitlines()
present = lambda s: any(json.loads(L)["pattern"].lower()==s.lower() for L in lines)
print("BSD in patterns? ", present("BSD"))
print("CCR in patterns? ", present("CCR"))

BSD in patterns?  True
CCR in patterns?  True


In [ ]:
# After building records (i.e., at the end):
print("Some REGION examples:")
for r in records:
    if r["category"] == "Region":
        print(" -", r["term"], "aka", r["aliases"][:5])

# Should find CCR/Regions now:
print("CCR present in spacy patterns?",
      any(p["pattern"] == "CCR" for p in spacy_patterns))
print("Core Central Region present?",
      any(p["pattern"] == "Core Central Region" for p in spacy_patterns))


Some REGION examples:
 - Core Central Region (CCR) aka ['CCR (Core Central Region)', 'Core Central Region', 'CCR']
 - Rest of Central Region (RCR) aka ['RCR (Rest of Central Region)', 'Rest of Central Region', 'RCR']
 - Outside Central Region (OCR) aka ['OCR (Outside Central Region)', 'Outside Central Region', 'OCR']
 - What is the Core Central Region (CCR)? aka ['What is the Core Central Region (CCR)?', 'Core Central Region', 'CCR']
CCR present in spacy patterns? True
Core Central Region present? True
